In [13]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score, classification_report

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from catboost import CatBoostClassifier
from sklearn.model_selection import LeaveOneGroupOut
from scipy.spatial import cKDTree

from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

import networkx as nx

import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv


/Users/polinamuradkhanova/Desktop/прикпитон/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv('grid_features.csv')
df.head()

,lat,lon,city,atm_count,sber_count,tinkoff_count,vtb_count,alfa_count,gazprom_count,raiff_count,...,poi_diversity_500m,poi_diversity_1000m,total_poi_500m,total_poi_1000m,retail_share_500m,residential_ratio_500m,orgs_500m,org_diversity_500m,orgs_1000m,org_diversity_1000m
0,54.965929,36.668509,Москва,0,0,0,0,0,0,0,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,54.965929,36.676464,Москва,0,0,0,0,0,0,0,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,54.965929,36.684419,Москва,0,0,0,0,0,0,0,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,54.965929,36.692373,Москва,0,0,0,0,0,0,0,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,54.965929,36.700328,Москва,0,0,0,0,0,0,0,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
df.columns

Index(['lat', 'lon', 'city', 'atm_count', 'sber_count', 'tinkoff_count',
       'vtb_count', 'alfa_count', 'gazprom_count', 'raiff_count', 'metro_500m',
       'metro_1000m', 'nearest_metro', 'bus_stops_500m', 'bus_stops_1000m',
       'nearest_bus_stops', 'malls_500m', 'malls_1000m', 'nearest_malls',
       'business_centres_500m', 'business_centres_1000m',
       'nearest_business_centres', 'universities_500m', 'universities_1000m',
       'nearest_universities', 'schools_500m', 'schools_1000m',
       'nearest_schools', 'hospitals_500m', 'hospitals_1000m',
       'nearest_hospitals', 'parks_500m', 'parks_1000m', 'nearest_parks',
       'residential_500m', 'residential_1000m', 'nearest_residential',
       'distance_to_centre', 'pyaterochka_500m', 'pyaterochka_1000m',
       'nearest_pyaterochka', 'magnit_500m', 'magnit_1000m', 'nearest_magnit',
       'perekrestok_500m', 'perekrestok_1000m', 'nearest_perekrestok',
       'diksi_500m', 'diksi_1000m', 'nearest_diksi', 'lenta_500m',
  

In [4]:
df.shape

(496447, 100)

In [3]:
df = pd.read_csv('grid_features.csv')
df['target'] = (df['atm_count'] > 0).astype(int)
drop_cols = ['atm_count', 'sber_count', 'tinkoff_count', 'vtb_count',
             'alfa_count', 'gazprom_count', 'raiff_count',
             'lat', 'lon', 'target']
feature_cols = [c for c in df.columns if c not in drop_cols and c != 'city']

X = df[feature_cols].copy()
y = df['target'].values
groups = df['city'].values
X = X.fillna(0)

In [6]:
df[df.atm_count >1]

,lat,lon,city,atm_count,sber_count,tinkoff_count,vtb_count,alfa_count,gazprom_count,raiff_count,...,poi_diversity_1000m,total_poi_500m,total_poi_1000m,retail_share_500m,residential_ratio_500m,orgs_500m,org_diversity_500m,orgs_1000m,org_diversity_1000m,target
6721,55.055760,38.744675,Москва,3,2,1,0,0,0,0,...,13,61.0,115.0,0.080645,0.629032,94.0,15.0,136.0,16.0,1
7368,55.064743,38.752629,Москва,2,1,1,0,0,0,0,...,16,47.0,87.0,0.104167,0.500000,68.0,11.0,118.0,14.0,1
8338,55.078218,38.760584,Москва,3,1,1,1,0,0,0,...,15,71.0,207.0,0.041667,0.791667,48.0,12.0,139.0,15.0,1
8657,55.082709,38.728765,Москва,2,1,1,0,0,0,0,...,6,7.0,10.0,0.125000,0.250000,5.0,2.0,10.0,2.0,1
8661,55.082709,38.760584,Москва,3,1,1,1,0,0,0,...,13,92.0,273.0,0.043011,0.720430,87.0,13.0,181.0,16.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
491485,55.867671,49.231176,Казань,2,2,0,0,0,0,0,...,16,145.0,326.0,0.041096,0.650685,48.0,11.0,133.0,14.0,1
491595,55.872163,48.879582,Казань,4,2,1,1,0,0,0,...,11,51.0,183.0,0.038462,0.557692,51.0,11.0,127.0,15.0,1
491751,55.876654,48.895564,Казань,2,2,0,0,0,0,0,...,13,73.0,126.0,0.040541,0.878378,18.0,8.0,48.0,14.0,1
493035,55.912587,49.311084,Казань,2,1,1,0,0,0,0,...,10,25.0,132.0,0.076923,0.500000,16.0,8.0,122.0,17.0,1


# бейзлайн

In [18]:


gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]
model = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100
)

model.fit(
    X_train,
    y_train
)
probs = model.predict_proba(X_test)[:, 1]
preds = (probs >= 0.5).astype(int)
f1 = f1_score(y_test, preds)

roc_auc = roc_auc_score(
    y_test,
    probs
)

pr_auc = average_precision_score(
    y_test,
    probs
)

print("\n===== METRICS =====")
print("F1:", round(f1, 4))
print("ROC-AUC:", round(roc_auc, 4))
print("PR-AUC:", round(pr_auc, 4))

print("\nClassification report:")
print(
    classification_report(
        y_test,
        preds
    )
)

feature_importance = pd.DataFrame({

    "feature": feature_cols,

    "importance": model.get_feature_importance()

}).sort_values(
    "importance",
    ascending=False
)
print("\nTop-20 features:")
print(
    feature_importance.head(20)
)

0:	total: 22ms	remaining: 11s
100:	total: 2.24s	remaining: 8.84s
200:	total: 4.22s	remaining: 6.28s
300:	total: 6.26s	remaining: 4.14s
400:	total: 8.29s	remaining: 2.04s
499:	total: 10.3s	remaining: 0us

===== METRICS =====
F1: 0.721
ROC-AUC: 0.9879
PR-AUC: 0.7938

Classification report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99     90965
           1       0.76      0.68      0.72      3674

    accuracy                           0.98     94639
   macro avg       0.87      0.84      0.86     94639
weighted avg       0.98      0.98      0.98     94639


Top-20 features:
                     feature  importance
66      nearest_gas_stations   35.889258
89       org_diversity_1000m   15.979263
88                orgs_1000m    7.445815
26       nearest_residential    5.726082
11  nearest_business_centres    4.705534
14      nearest_universities    4.245476
27        distance_to_centre    3.579094
5          nearest_bus_stops    2.6678

# mlp

In [4]:
groups = df["city"].values

gss = GroupShuffleSplit(n_splits=1,
    test_size=0.2,
    random_state=42)

train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y[train_idx]
y_test = y[test_idx]
print("Train size:", X_train.shape)
print("Test size:", X_test.shape)
print("Train target mean:", y_train.mean())
print("Test target mean:", y_test.mean())

Train size: (401808, 90)
Test size: (94639, 90)
Train target mean: 0.0064458646915939955
Test target mean: 0.038821204788723467


In [5]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)


train_dataset = TensorDataset(X_train_tensor, y_train_tensor)

train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True
)


class MLPClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.2),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)


device = "cuda" if torch.cuda.is_available() else "cpu"

model = MLPClassifier(input_dim=X_train_scaled.shape[1]).to(device)


num_pos = y_train.sum()
num_neg = len(y_train) - num_pos

pos_weight = torch.tensor([num_neg / num_pos], dtype=torch.float32).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)
epochs = 50

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for batch_X, batch_y in train_loader:

        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()

        logits = model(batch_X)
        loss = criterion(logits, batch_y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {avg_loss:.4f}")


model.eval()

with torch.no_grad():
    test_logits = model(X_test_tensor.to(device))
    test_probs = torch.sigmoid(test_logits).cpu().numpy().flatten()
test_preds = (test_probs >= 0.5).astype(int)

f1 = f1_score(y_test, test_preds)
roc_auc = roc_auc_score(y_test, test_probs)
pr_auc = average_precision_score(y_test, test_probs)
print("F1:", round(f1, 4))
print("ROC-AUC:", round(roc_auc, 4))
print("PR-AUC:", round(pr_auc, 4))

print("\nClassification report:")
print(classification_report(y_test, test_preds))

Epoch 5/50, Loss: 0.1391
Epoch 10/50, Loss: 0.1358
Epoch 15/50, Loss: 0.1306
Epoch 20/50, Loss: 0.1254
Epoch 25/50, Loss: 0.1210
Epoch 30/50, Loss: 0.1259
Epoch 35/50, Loss: 0.1244
Epoch 40/50, Loss: 0.1188
Epoch 45/50, Loss: 0.1193
Epoch 50/50, Loss: 0.1205
F1: 0.4615
ROC-AUC: 0.982
PR-AUC: 0.717

Classification report:
              precision    recall  f1-score   support

           0       1.00      0.91      0.95     90965
           1       0.30      0.98      0.46      3674

    accuracy                           0.91     94639
   macro avg       0.65      0.94      0.71     94639
weighted avg       0.97      0.91      0.93     94639



## hex2vec

In [ ]:
import h3

def latlon_to_h3(lat, lon, resolution=9):

    if hasattr(h3, "latlng_to_cell"):
        return h3.latlng_to_cell(lat, lon, resolution)
    else:
        return h3.geo_to_h3(lat, lon, resolution)


def h3_neighbors(cell, k=1):
    """
    Соседние гексагоны вокруг текущего.
    k=1 — ближайшее кольцо.
    """
    if hasattr(h3, "grid_disk"):
        return list(h3.grid_disk(cell, k))
    else:
        return list(h3.k_ring(cell, k))


H3_RESOLUTION = 9

df["h3_cell"] = df.apply(
    lambda row: latlon_to_h3(row["lat"], row["lon"], H3_RESOLUTION),
    axis=1
)

drop_cols = [
    "atm_count",
    "sber_count",
    "tinkoff_count",
    "vtb_count",
    "alfa_count",
    "gazprom_count",
    "raiff_count",
    "lat",
    "lon",
    "target",
    "city",
    "h3_cell"
]

feature_cols = [c for c in df.columns if c not in drop_cols]

X_base = df[feature_cols].copy().fillna(0)
y = df["target"].values
groups = df["city"].values

hex_features = (
    pd.concat([df[["h3_cell"]], X_base], axis=1)
    .groupby("h3_cell")[feature_cols]
    .mean()
)

hex_to_features = {
    h: hex_features.loc[h].values
    for h in hex_features.index
}

feature_dim = len(feature_cols)


def get_neighbor_mean_features(cell, k=1):
    """
    Для каждой клетки берём средние признаки соседних h3-гексагонов.
    """
    neighbors = h3_neighbors(cell, k=k)

    vectors = [
        hex_to_features[n]
        for n in neighbors
        if n in hex_to_features
    ]

    if len(vectors) == 0:
        return np.zeros(feature_dim)

    return np.mean(vectors, axis=0)


neighbor_features = np.vstack(
    df["h3_cell"].apply(lambda cell: get_neighbor_mean_features(cell, k=1)).values
)

neighbor_feature_cols = [f"neighbor_mean_{c}" for c in feature_cols]

X_neighbor = pd.DataFrame(
    neighbor_features,
    columns=neighbor_feature_cols,
    index=df.index
)
X_for_embedding = pd.concat([X_base, X_neighbor], axis=1).fillna(0)

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(gss.split(X_for_embedding, y, groups=groups))

X_emb_train = X_for_embedding.iloc[train_idx]
X_emb_test = X_for_embedding.iloc[test_idx]

y_train = y[train_idx]
y_test = y[test_idx]
emb_scaler = StandardScaler()

X_emb_train_scaled = emb_scaler.fit_transform(X_emb_train)
X_emb_test_scaled = emb_scaler.transform(X_emb_test)


class HexAutoEncoder(nn.Module):
    def __init__(self, input_dim, embedding_dim=16):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, embedding_dim)
        )

        self.decoder = nn.Sequential(
            nn.Linear(embedding_dim, 64),
            nn.ReLU(),

            nn.Linear(64, 128),
            nn.ReLU(),

            nn.Linear(128, input_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        reconstructed = self.decoder(z)
        return reconstructed

    def encode(self, x):
        return self.encoder(x)


device = "cuda" if torch.cuda.is_available() else "cpu"

embedding_dim = 16

autoencoder = HexAutoEncoder(
    input_dim=X_emb_train_scaled.shape[1],
    embedding_dim=embedding_dim
).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(autoencoder.parameters(), lr=0.001, weight_decay=1e-5)

X_ae_train_tensor = torch.tensor(X_emb_train_scaled, dtype=torch.float32)

ae_dataset = TensorDataset(X_ae_train_tensor)
ae_loader = DataLoader(ae_dataset, batch_size=256, shuffle=True)

ae_epochs = 40

for epoch in range(ae_epochs):
    autoencoder.train()
    total_loss = 0

    for (batch_X,) in ae_loader:
        batch_X = batch_X.to(device)

        optimizer.zero_grad()

        reconstructed = autoencoder(batch_X)
        loss = criterion(reconstructed, batch_X)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if (epoch + 1) % 5 == 0:
        print(f"AutoEncoder Epoch {epoch + 1}/{ae_epochs}, Loss: {total_loss / len(ae_loader):.4f}")


autoencoder.eval()

with torch.no_grad():
    train_embeddings = autoencoder.encode(
        torch.tensor(X_emb_train_scaled, dtype=torch.float32).to(device)
    ).cpu().numpy()

    test_embeddings = autoencoder.encode(
        torch.tensor(X_emb_test_scaled, dtype=torch.float32).to(device)
    ).cpu().numpy()


emb_cols = [f"hex_emb_{i}" for i in range(embedding_dim)]

X_train_final = pd.concat(
    [
        X_base.iloc[train_idx].reset_index(drop=True),
        pd.DataFrame(train_embeddings, columns=emb_cols)
    ],
    axis=1
)

X_test_final = pd.concat(
    [
        X_base.iloc[test_idx].reset_index(drop=True),
        pd.DataFrame(test_embeddings, columns=emb_cols)
    ],
    axis=1
)


final_scaler = StandardScaler()

X_train_scaled = final_scaler.fit_transform(X_train_final)
X_test_scaled = final_scaler.transform(X_test_final)

class MLPClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.2),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)


X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)

train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True
)

model = MLPClassifier(input_dim=X_train_scaled.shape[1]).to(device)

num_pos = y_train.sum()
num_neg = len(y_train) - num_pos

pos_weight = torch.tensor([num_neg / num_pos], dtype=torch.float32).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

epochs = 50

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch_X, batch_y in train_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()

        logits = model(batch_X)
        loss = criterion(logits, batch_y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if (epoch + 1) % 5 == 0:
        print(f"MLP Epoch {epoch + 1}/{epochs}, Loss: {total_loss / len(train_loader):.4f}")


model.eval()

with torch.no_grad():
    test_logits = model(X_test_tensor.to(device))
    test_probs = torch.sigmoid(test_logits).cpu().numpy().flatten()

test_preds_05 = (test_probs >= 0.5).astype(int)

f1_05 = f1_score(y_test, test_preds_05)
roc_auc = roc_auc_score(y_test, test_probs)
pr_auc = average_precision_score(y_test, test_probs)

print("\n===== METRICS WITH THRESHOLD 0.5 =====")
print("F1:", round(f1_05, 4))
print("ROC-AUC:", round(roc_auc, 4))
print("PR-AUC:", round(pr_auc, 4))

print("\nClassification report:")
print(classification_report(y_test, test_preds_05))
thresholds = np.arange(0.05, 0.95, 0.01)

best_f1 = 0
best_threshold = 0.5

for threshold in thresholds:
    preds = (test_probs >= threshold).astype(int)
    current_f1 = f1_score(y_test, preds)

    if current_f1 > best_f1:
        best_f1 = current_f1
        best_threshold = threshold

best_preds = (test_probs >= best_threshold).astype(int)

print("\n===== METRICS WITH BEST THRESHOLD =====")
print("Best threshold:", round(best_threshold, 3))
print("F1:", round(best_f1, 4))
print("ROC-AUC:", round(roc_auc, 4))
print("PR-AUC:", round(pr_auc, 4))

print("\nClassification report with best threshold:")
print(classification_report(y_test, best_preds))

AutoEncoder Epoch 5/40, Loss: 0.1434
AutoEncoder Epoch 10/40, Loss: 0.1077
AutoEncoder Epoch 15/40, Loss: 0.0908
AutoEncoder Epoch 20/40, Loss: 0.0871
AutoEncoder Epoch 25/40, Loss: 0.0809
AutoEncoder Epoch 30/40, Loss: 0.0732
AutoEncoder Epoch 35/40, Loss: 0.0721
AutoEncoder Epoch 40/40, Loss: 0.0706
MLP Epoch 5/50, Loss: 0.1382
MLP Epoch 10/50, Loss: 0.1295
MLP Epoch 15/50, Loss: 0.1289
MLP Epoch 20/50, Loss: 0.1261
MLP Epoch 25/50, Loss: 0.1277
MLP Epoch 30/50, Loss: 0.1311
MLP Epoch 35/50, Loss: 0.1231
MLP Epoch 40/50, Loss: 0.1203
MLP Epoch 45/50, Loss: 0.1186
MLP Epoch 50/50, Loss: 0.1180

===== METRICS WITH THRESHOLD 0.5 =====
F1: 0.4388
ROC-AUC: 0.9849
PR-AUC: 0.724

Classification report:
              precision    recall  f1-score   support

           0       1.00      0.90      0.95     90965
           1       0.28      0.98      0.44      3674

    accuracy                           0.90     94639
   macro avg       0.64      0.94      0.69     94639
weighted avg       0.

# GraphSAGE

In [ ]:
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups)
)

scaler = StandardScaler()

X_train = scaler.fit_transform(
    X.iloc[train_idx]
)

X_test = scaler.transform(
    X.iloc[test_idx]
)

X_scaled = np.zeros_like(X.values)

X_scaled[train_idx] = X_train
X_scaled[test_idx] = X_test
coords = df[["lat", "lon"]].values

knn = NearestNeighbors(
    n_neighbors=6,
    metric="euclidean"
)

knn.fit(coords)

distances, indices = knn.kneighbors(coords)

edges = []

for i in range(len(df)):

    neighbors = indices[i][1:]

    for neighbor in neighbors:

        edges.append([i, neighbor])
        edges.append([neighbor, i])

edge_index = torch.tensor(
    edges,
    dtype=torch.long
).t().contiguous()

x_tensor = torch.tensor(
    X_scaled,
    dtype=torch.float32
)

y_tensor = torch.tensor(
    y,
    dtype=torch.float32
)

train_mask = torch.zeros(
    len(df),
    dtype=torch.bool
)

test_mask = torch.zeros(
    len(df),
    dtype=torch.bool
)

train_mask[train_idx] = True
test_mask[test_idx] = True

data = Data(
    x=x_tensor,
    edge_index=edge_index,
    y=y_tensor
)

data.train_mask = train_mask
data.test_mask = test_mask

class GraphSAGE(torch.nn.Module):

    def __init__(self, in_channels):

        super().__init__()

        self.conv1 = SAGEConv(
            in_channels,
            64
        )

        self.conv2 = SAGEConv(
            64,
            32
        )

        self.lin = torch.nn.Linear(
            32,
            1
        )

    def forward(self, x, edge_index):

        x = self.conv1(
            x,
            edge_index
        )

        x = F.relu(x)

        x = F.dropout(
            x,
            p=0.3,
            training=self.training
        )

        x = self.conv2(
            x,
            edge_index
        )

        x = F.relu(x)

        x = self.lin(x)

        return x.squeeze()


device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = GraphSAGE(
    in_channels=X.shape[1]
).to(device)

data = data.to(device)
num_pos = y[train_idx].sum()
num_neg = len(train_idx) - num_pos

pos_weight = torch.tensor(
    [num_neg / num_pos],
    dtype=torch.float32
).to(device)

criterion = torch.nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.005,
    weight_decay=1e-4
)

epochs = 100
for epoch in range(epochs):

    model.train()

    optimizer.zero_grad()

    out = model(
        data.x,
        data.edge_index
    )

    loss = criterion(
        out[data.train_mask],
        data.y[data.train_mask]
    )

    loss.backward()

    optimizer.step()

    if (epoch + 1) % 10 == 0:

        print(
            f"Epoch {epoch+1}, "
            f"Loss: {loss.item():.4f}"
        )
model.eval()

with torch.no_grad():

    logits = model(
        data.x,
        data.edge_index
    )

    probs = torch.sigmoid(
        logits[data.test_mask]
    ).cpu().numpy()

preds = (
    probs >= 0.5
).astype(int)

y_true = y[test_idx]

f1 = f1_score(
    y_true,
    preds
)

roc_auc = roc_auc_score(
    y_true,
    probs
)

pr_auc = average_precision_score(
    y_true,
    probs
)

print("\n===== METRICS =====")
print("F1:", round(f1, 4))
print("ROC-AUC:", round(roc_auc, 4))
print("PR-AUC:", round(pr_auc, 4))

Epoch 10, Loss: 0.3156
Epoch 20, Loss: 0.1711
Epoch 30, Loss: 0.1511
Epoch 40, Loss: 0.1434
Epoch 50, Loss: 0.1394
Epoch 60, Loss: 0.1350
Epoch 70, Loss: 0.1328
Epoch 80, Loss: 0.1312
Epoch 90, Loss: 0.1284
Epoch 100, Loss: 0.1272

===== METRICS =====
F1: 0.3649
ROC-AUC: 0.9849
PR-AUC: 0.7258


Модель CatBoost показала наилучшее качество среди всех рассмотренных подходов. Наиболее значимыми признаками оказались расстояние до метро, плотность жилой застройки и близость крупных ритейлеров. MLP и GraphSAGE продемонстрировали высокие значения ROC-AUC, однако существенно уступили по F1-score. Использование hex2vec-подобных пространственных эмбеддингов улучшило качество MLP, но метрики все равно ниже бейзлайна. 